In [ ]:
import pathlib

import matplotlib.pyplot as plt
import networkx
import numpy
import pandas
import powerlaw
import seaborn

In [ ]:
def get_adjacency_matrix(sheet_name: str, path: pathlib.Path) -> tuple:
    names, adjacency_matrix = numpy.hsplit(
        pandas.read_excel(io=path, sheet_name=sheet_name).to_numpy(), [1]
    )
    names = names.reshape(names.size)
    adjacency_matrix = adjacency_matrix.astype(numpy.int32)
    return names, adjacency_matrix


def write_dataframe_to_file(
    dataframe: pandas.DataFrame,
    sheet_name: str,
    path: pathlib.Path,
) -> None:

    if path.exists():
        with pandas.ExcelWriter(
            path=path,
            engine="openpyxl",
            mode="a",
            if_sheet_exists="replace",
        ) as writer:
            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )
    else:
        with pandas.ExcelWriter(
            path=path,
            engine="openpyxl",
            mode="w",
        ) as writer:
            dataframe.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )

In [ ]:
dataset_path = pathlib.Path("../../dataset/migration_matrix.xlsx")
report_path = pathlib.Path("../../output/report.xlsx")

county_sheet = "County"
province_sheet = "Province"

self_loop = False

In [ ]:
report_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
province_names, province_adjacency_matrix = get_adjacency_matrix(
    sheet_name=province_sheet, path=dataset_path
)
county_names, county_adjacency_matrix = get_adjacency_matrix(
    sheet_name=county_sheet, path=dataset_path
)

if self_loop is False:
    numpy.fill_diagonal(province_adjacency_matrix, val=0)
    numpy.fill_diagonal(county_adjacency_matrix, val=0)

province_graph = networkx.from_numpy_array(
    A=province_adjacency_matrix, create_using=networkx.DiGraph
)
networkx.relabel_nodes(
    G=province_graph, mapping={i: j for i, j in enumerate(province_names)}, copy=False
)

county_graph = networkx.from_numpy_array(
    A=county_adjacency_matrix, create_using=networkx.DiGraph
)
networkx.relabel_nodes(
    G=county_graph, mapping={i: j for i, j in enumerate(county_names)}, copy=False
);

# Network Information

In [ ]:
def network_basis_information(network: networkx.DiGraph) -> pandas.DataFrame:

    network_measures = {
        "Number of Nodes": network.number_of_nodes(),
        "Number of Edges": network.number_of_edges(),
        "Density": networkx.density(network),
        "Diameter": networkx.diameter(network),
        "Average Clustering Coefficient": networkx.average_clustering(network),
        "Degree Assortativity Coefficient": networkx.degree_assortativity_coefficient(
            network, x="out", y="in", weight="weight"
        ),
        "Strongly Connected": networkx.is_strongly_connected(network),
        "Weakly Connected": networkx.is_weakly_connected(network),
    }

    dataframe = pandas.DataFrame(columns=["Measure", "Value"])

    for key, value in network_measures.items():
        dataframe = pandas.concat(
            [
                dataframe,
                pandas.DataFrame({"Measure": [key], "Value": [value]}),
            ]
        )

    return dataframe

In [ ]:
write_dataframe_to_file(
    dataframe=network_basis_information(network=province_graph),
    sheet_name="Prv_basis_info",
    path=report_path,
)

write_dataframe_to_file(
    dataframe=network_basis_information(network=county_graph),
    sheet_name="Cnt_basis_info",
    path=report_path,
)

# Degree Information

In [ ]:
def degree_information(
    names: list, adjacency_matrix: numpy.ndarray, first_column_name: str
) -> pandas.DataFrame:

    degree_info = [
        [
            names[i],
            numpy.sum(adjacency_matrix, axis=0)[i],
            numpy.sum(adjacency_matrix, axis=1)[i],
            numpy.count_nonzero(adjacency_matrix[:, i]),
            numpy.count_nonzero(adjacency_matrix[i, :]),
            numpy.log10(numpy.sum(adjacency_matrix, axis=0)[i]),
            numpy.log10(numpy.sum(adjacency_matrix, axis=1)[i]),
            numpy.log10(numpy.count_nonzero(adjacency_matrix[:, i])),
            numpy.log10(numpy.count_nonzero(adjacency_matrix[i, :])),
        ]
        for i in range(len(names))
    ]

    return pandas.DataFrame(
        columns=[
            first_column_name,
            "Weighted In-Degree",
            "Weighted Out-Degree",
            "Unweighted In-Degree",
            "Unweighted Out-Degree",
            "Weighted In-Degree Log",
            "Weighted Out-Degree Log",
            "Unweighted In-Degree Log",
            "Unweighted Out-Degree Log",
        ],
        data=degree_info,
    )

In [ ]:
province_degree_info = degree_information(
    names=province_names,
    adjacency_matrix=province_adjacency_matrix,
    first_column_name="Province",
)
county_degree_info = degree_information(
    names=county_names,
    adjacency_matrix=county_adjacency_matrix,
    first_column_name="County",
)

write_dataframe_to_file(
    dataframe=province_degree_info,
    sheet_name="Prv_nodes_info",
    path=report_path,
)
write_dataframe_to_file(
    dataframe=county_degree_info,
    sheet_name="Cnt_nodes_info",
    path=report_path,
)

### Plot Degree Information

In [ ]:
scatter_color = "purple"

plt.rcParams["font.family"] = "DejaVu Serif"

In [ ]:
network_type, degree_info = "Province", province_degree_info

for degree_type in [
    "Weighted In-Degree Log",
    "Weighted Out-Degree Log",
]:
    degree_info = degree_info.sort_values(by=degree_type, ascending=False)

    fig, axes = plt.subplots()

    axes.plot(
        degree_info[network_type],
        degree_info[degree_type],
        marker="o",
        color=scatter_color,
        linestyle="--",
        linewidth=0.8,
    )
    axes.set_xlabel(network_type)
    axes.set_ylabel(degree_type + " Scale")
    axes.tick_params(axis="x", rotation=90)

    fig.tight_layout()

    plt.savefig(
        f"../../output/{network_type.lower()}_{degree_type.lower().replace(' ', '_')}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()

In [ ]:
network_type, degree_info = "County", county_degree_info

for degree_type in [
    "Weighted In-Degree Log",
    "Weighted Out-Degree Log",
    "Unweighted In-Degree Log",
    "Unweighted Out-Degree Log",
]:
    degree_info = degree_info.sort_values(by=degree_type, ascending=False)

    fig, axes = plt.subplots()

    axes.plot(
        degree_info[network_type][::12],
        degree_info[degree_type][::12],
        marker="o",
        color=scatter_color,
        linestyle="--",
        linewidth=0.8,
    )
    axes.set_xlabel(network_type)
    axes.set_ylabel(degree_type + " Scale")
    axes.tick_params(axis="x", rotation=90)

    fig.tight_layout()

    plt.savefig(
        f"../../output/{network_type.lower()}_{degree_type.lower().replace(' ', '_')}.png",
        bbox_inches="tight",
        dpi=300,
    )
    plt.show()

# Centrality

In [ ]:
eigenvector_max_iterations = 100
eigenvector_tolerance = 1e-06
eigenvector_starting_value = None
eigenvector_weight = "weight"

closeness_end_node = None
closeness_distance = None
closeness_wf_improved = True

betweenness_k = None
betweenness_normalized = True
betweenness_weight = None
betweenness_endpoints = False
betweenness_seed = None

pagerank_alpha = 0.85
pagerank_personalization = None
pagerank_max_iterations = 100
pagerank_tolerance = 1e-06
pagerank_starting_value = None
pagerank_weight = "weight"
pagerank_dangling = None

In [ ]:
def calculate_centrality(
    network: networkx.DiGraph, first_column_name: str
) -> pandas.DataFrame:

    network_centralities = {
        "In-Degree Centrality": networkx.in_degree_centrality(network).values(),
        "Out-Degree Centrality": networkx.out_degree_centrality(network).values(),
        "Eigenvector Centrality": networkx.eigenvector_centrality(
            network,
            max_iter=eigenvector_max_iterations,
            tol=eigenvector_tolerance,
            nstart=eigenvector_starting_value,
            weight="weight",
        ).values(),
        "Closeness Centrality": networkx.closeness_centrality(
            network,
            u=closeness_end_node,
            distance=closeness_distance,
            wf_improved=closeness_wf_improved,
        ).values(),
        "Betweenness Centrality": networkx.betweenness_centrality(
            network,
            k=betweenness_k,
            normalized=betweenness_normalized,
            weight=betweenness_weight,
            endpoints=betweenness_endpoints,
            seed=betweenness_seed,
        ).values(),
        "PageRank": networkx.pagerank(
            network,
            alpha=pagerank_alpha,
            personalization=pagerank_personalization,
            max_iter=pagerank_max_iterations,
            tol=pagerank_tolerance,
            nstart=pagerank_starting_value,
            weight=pagerank_weight,
            dangling=pagerank_dangling,
        ).values(),
    }

    dataframe = pandas.DataFrame(columns=[first_column_name], data=list(network.nodes))

    for key, value in network_centralities.items():
        dataframe[key] = value
    return dataframe

In [ ]:
write_dataframe_to_file(
    dataframe=calculate_centrality(
        network=province_graph, first_column_name="Province"
    ),
    sheet_name="Prv_cntr_info",
    path=report_path,
)

write_dataframe_to_file(
    dataframe=calculate_centrality(network=county_graph, first_column_name="County"),
    sheet_name="Cnt_cntr_info",
    path=report_path,
)

# Power-law & Degree Correlation

In [ ]:
correlation_method = "pearson"

In [ ]:
network_type, degree_info = "Province", province_degree_info

for degree_type in [
    "Weighted In-Degree",
    "Weighted Out-Degree",
]:
    results = powerlaw.Fit(
        degree_info.loc[:, [degree_type]]
        .values.reshape(len(degree_info.index))
        .tolist(),
        discrete=True,
    )
    print(
        network_type,
        degree_type,
        f"\n\t alpha = {results.power_law.alpha} \n\t X min = {results.power_law.xmin}",
    )

In [ ]:
network_type, degree_info = "County", county_degree_info

for degree_type in [
    "Weighted In-Degree",
    "Weighted Out-Degree",
    "Unweighted In-Degree",
    "Unweighted Out-Degree",
]:
    results = powerlaw.Fit(
        degree_info.loc[:, [degree_type]]
        .values.reshape(len(degree_info.index))
        .tolist(),
        discrete=True,
    )
    print(
        network_type,
        degree_type,
        f"\n\t alpha = {results.power_law.alpha} \n\t X min = {results.power_law.xmin}",
    )

In [ ]:
province_degree_correlation = province_degree_info.loc[
    :,
    [
        "Weighted In-Degree",
        "Weighted Out-Degree",
    ],
].corr(method=correlation_method)

round(province_degree_correlation, 2)

seaborn.heatmap(
    province_degree_correlation,
    cmap=seaborn.cubehelix_palette(as_cmap=True),
    annot=True,
)

plt.savefig(
    "../../output/province_degree_correlation.png",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
county_degree_correlation = county_degree_info.loc[
    :,
    [
        "Weighted In-Degree",
        "Weighted Out-Degree",
        "Unweighted In-Degree",
        "Unweighted Out-Degree",
    ],
].corr(method=correlation_method)

round(county_degree_correlation, 2)

seaborn.heatmap(
    county_degree_correlation, cmap=seaborn.cubehelix_palette(as_cmap=True), annot=True
)

plt.savefig(
    "../../output/county_degree_correlation.png",
    bbox_inches="tight",
    dpi=300,
)

# Gephi

In [ ]:
gephi_file_path = pathlib.Path("../../output/gephi.xlsx")

In [ ]:
for network_type, names, adj_matrix in [
    ("Province", province_names, province_adjacency_matrix),
    ("County", county_names, county_adjacency_matrix),
]:
    gephi_dataframe = degree_information(
        names=names, adjacency_matrix=adj_matrix, first_column_name="ID"
    )
    gephi_dataframe.insert(loc=1, column="Label", value=gephi_dataframe["ID"])
    write_dataframe_to_file(
        dataframe=gephi_dataframe, sheet_name=network_type, path=gephi_file_path
    )